# Projeto #1: Churn Prediction

## Projeto #1: Churn Prediction

**Domínio:** SaaS/Streaming/Telecom — qualquer empresa com clientes recorrentes
**Pergunta:** Qual cliente vai cancelar? Por quê? Como reter?
**Conceitos cobertos:** Structured Outputs (S2), RAG tradicional + adaptativo (S3-4),
Tool Use (S5), Agentic Loop com stopping rule (S5-6), LangGraph linear (S6-7),
Confidence Scoring (S7), Observability (S8)

**Sobre realismo:** este notebook chama a **API real da Anthropic** quando
você define `ANTHROPIC_API_KEY` no ambiente. Sem a chave, ele cai
automaticamente num mock determinístico — assim dá pra estudar a arquitetura
inteira sem gastar um centavo, e trocar pra chamadas reais só definindo a
variável de ambiente, sem mudar nenhuma linha de código.

In [1]:
!pip install -q langgraph pydantic anthropic

import os
import random
from typing import Literal, Optional
from pydantic import BaseModel, Field

### 1. Schema estruturado (Semana 2)

**Por que esta classe existe:** sem um schema, a saída do LLM é uma string
livre que você teria que fazer parsing manual (frágil). `ChurnPrediction` é o
contrato: toda predição do agente tem exatamente esses 5 campos, com tipos e
ranges validados — se o LLM (real ou mock) devolver algo fora do formato, o
Pydantic levanta `ValidationError` na hora, não em produção 3 semanas depois.

`CustomerState` é o "estado" que trafega pelos nós do grafo (seção 6) — cada
nó lê e enriquece esse objeto.

In [2]:
class ChurnPrediction(BaseModel):
    customer_id: str
    will_churn: bool
    confidence: float = Field(ge=0.0, le=1.0)
    risk_factors: list[str]
    recommended_action: str

class CustomerState(BaseModel):
    customer_id: str
    tenure_months: int
    monthly_spend: float
    support_tickets_90d: int
    engagement_score: float  # 0-1
    similar_churned: list[dict] = []
    prediction: Optional[ChurnPrediction] = None
    strategy_attempts: int = 0
    verified: bool = False

**Resultado esperado:** nada acontece ainda — são só as definições de schema.
O teste real de validação acontece quando alguma instância for criada com
tipo errado (o Pydantic barra na hora).

### 2. Dados sintéticos (substituem o CSV de 10k clientes)

**Por que esta função existe:** o dataset de 10k clientes reais mencionado no
plano do curso não existe (é fictício por natureza — dado de cliente real não
entra num repositório público). `generate_customers` cria um substituto
determinístico (mesma seed = mesmos clientes sempre) grande o suficiente pra
testar o pipeline, mas pequeno o suficiente pra rodar em segundos.

In [3]:
def generate_customers(n: int = 20, seed: int = 42) -> list[CustomerState]:
    random.seed(seed)
    customers = []
    for i in range(n):
        customers.append(CustomerState(
            customer_id=f"cust_{i:04d}",
            tenure_months=random.randint(1, 48),
            monthly_spend=round(random.uniform(20, 500), 2),
            support_tickets_90d=random.randint(0, 8),
            engagement_score=round(random.uniform(0.1, 1.0), 2),
        ))
    return customers

customers = generate_customers()
print(f"✓ {len(customers)} clientes sintéticos gerados")
print(customers[0])

✓ 20 clientes sintéticos gerados
customer_id='cust_0000' tenure_months=41 monthly_spend=73.44 support_tickets_90d=4 engagement_score=0.32 similar_churned=[] prediction=None strategy_attempts=0 verified=False


**Resultado esperado:** imprime `✓ 20 clientes sintéticos gerados` e o
primeiro cliente (`cust_0000`, com tenure/spend/tickets/engagement
aleatórios mas reproduzíveis pela seed 42).

### 3. RAG leve — clientes similares que já cancelaram (Semana 3-4)

**Por que esta função existe:** antes de prever, é útil saber "clientes
parecidos com esse já cancelaram? por quê?" — isso é RAG. A parte
*adaptativa* está no `if needs_retrieval`: só gastamos o custo de busca
quando o cliente já mostra sinal de risco (tickets altos ou engajamento
baixo). Pra clientes saudáveis, pular o retrieval economiza tempo e,
numa implementação real com embeddings, dinheiro.

In [4]:
def retrieve_similar_churned(customer: CustomerState, churn_history: list[dict]) -> list[dict]:
    """Adaptive RAG: só retrieva se o cliente tiver sinais de risco.
    Evita custo de retrieval quando não precisa (10x mais eficiente)."""
    needs_retrieval = customer.support_tickets_90d >= 3 or customer.engagement_score < 0.4
    if not needs_retrieval:
        return []

    def distance(c):
        return abs(c["tenure_months"] - customer.tenure_months) + \
               abs(c["engagement_score"] - customer.engagement_score) * 10
    return sorted(churn_history, key=distance)[:3]

CHURN_HISTORY = [
    {"customer_id": "hist_01", "tenure_months": 3, "engagement_score": 0.2, "reason": "onboarding ruim"},
    {"customer_id": "hist_02", "tenure_months": 14, "engagement_score": 0.3, "reason": "concorrente mais barato"},
    {"customer_id": "hist_03", "tenure_months": 2, "engagement_score": 0.15, "reason": "não usou o produto"},
]

**Resultado esperado:** nenhum output visível ainda (função + dados de
apoio) — ela é chamada dentro do grafo, na seção 6.

### 4. Tool: consultar histórico de suporte (Semana 5)

**Por que esta função existe:** é uma *tool* — uma função que o agente chama
pra buscar informação que não está no `CustomerState`. Em produção, isso
seria uma chamada real a um sistema de tickets (Zendesk, Intercom); aqui é
determinística (seed baseada no `customer_id`) pra ser reproduzível.

In [5]:
def tool_query_support_history(customer_id: str) -> dict:
    """Tool que o agente chama pra buscar contexto adicional."""
    random.seed(hash(customer_id) % 1000)
    return {
        "customer_id": customer_id,
        "avg_resolution_hours": round(random.uniform(1, 48), 1),
        "satisfaction_score": round(random.uniform(1, 5), 1),
    }

### 5. Chamada ao LLM — real com fallback pro mock (Semana 1, 5, 6, 7)

**Por que esta função existe:** é o coração do agente — decide se o cliente
vai cancelar. `call_claude_structured` é o wrapper que **realmente chama a
Anthropic API** (usando `tool_use` pra forçar a saída no formato do
`ChurnPrediction`) quando `ANTHROPIC_API_KEY` existe no ambiente. Sem chave,
`heuristic_prediction` assume — uma heurística determinística que simula o
que um LLM razoável diria, calculada a partir dos mesmos sinais que um
prompt real usaria (engajamento, tickets, tenure, histórico similar).

In [6]:
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
USE_REAL_LLM = bool(ANTHROPIC_API_KEY)

if USE_REAL_LLM:
    import anthropic
    _client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

CHURN_TOOL_SCHEMA = {
    "name": "record_churn_prediction",
    "description": "Registra a predição de churn estruturada",
    "input_schema": {
        "type": "object",
        "properties": {
            "will_churn": {"type": "boolean"},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "risk_factors": {"type": "array", "items": {"type": "string"}},
            "recommended_action": {"type": "string"},
        },
        "required": ["will_churn", "confidence", "risk_factors", "recommended_action"],
    },
}

def call_claude_structured(customer: CustomerState, strategy_prompt: str) -> Optional[dict]:
    """Chama a Anthropic API de verdade, forçando saída estruturada via
    tool_use. Retorna None se não houver API key (caller usa o fallback)."""
    if not USE_REAL_LLM:
        return None
    response = _client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        tools=[CHURN_TOOL_SCHEMA],
        tool_choice={"type": "tool", "name": "record_churn_prediction"},
        messages=[{
            "role": "user",
            "content": (
                f"{strategy_prompt}\n\nDados do cliente:\n"
                f"- tenure: {customer.tenure_months} meses\n"
                f"- gasto mensal: ${customer.monthly_spend}\n"
                f"- tickets de suporte (90d): {customer.support_tickets_90d}\n"
                f"- engagement score: {customer.engagement_score}\n"
                f"- clientes similares que já cancelaram: {customer.similar_churned}"
            ),
        }],
    )
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return None

def heuristic_prediction(customer: CustomerState, strategy: str) -> ChurnPrediction:
    """Fallback determinístico — usado quando não há ANTHROPIC_API_KEY.
    Calcula risco a partir dos mesmos sinais que um prompt real usaria."""
    base_risk = (
        (1 - customer.engagement_score) * 0.5
        + min(customer.support_tickets_90d / 8, 1) * 0.3
        + (1 if customer.tenure_months < 3 else 0) * 0.2
    )
    strategy_boost = {"heuristic": 0.0, "cot": 0.08, "rag_augmented": 0.15}[strategy]
    confidence = min(0.95, base_risk * 0.6 + 0.3 + strategy_boost)

    risk_factors = []
    if customer.engagement_score < 0.4:
        risk_factors.append("baixo engajamento")
    if customer.support_tickets_90d >= 3:
        risk_factors.append("muitos tickets de suporte")
    if customer.tenure_months < 3:
        risk_factors.append("cliente novo (onboarding crítico)")
    if customer.similar_churned:
        risk_factors.append(f"{len(customer.similar_churned)} clientes similares já cancelaram")

    return ChurnPrediction(
        customer_id=customer.customer_id,
        will_churn=base_risk > 0.5,
        confidence=round(confidence, 2),
        risk_factors=risk_factors or ["sem sinais fortes de risco"],
        recommended_action="oferecer desconto + onboarding assistido" if base_risk > 0.5 else "monitorar",
    )

STRATEGY_PROMPTS = {
    "heuristic": "Analise os dados do cliente e avalie o risco de churn.",
    "cot": "Pense passo a passo sobre os dados do cliente antes de decidir o risco de churn.",
    "rag_augmented": "Considere os clientes similares que já cancelaram e analise o risco de churn deste cliente.",
}

def mock_llm_call(customer: CustomerState, strategy: str) -> ChurnPrediction:
    """Ponto único de decisão: tenta a API real, cai pro heurístico se
    não tiver chave ou se a chamada falhar."""
    result = call_claude_structured(customer, STRATEGY_PROMPTS[strategy])
    if result is not None:
        return ChurnPrediction(customer_id=customer.customer_id, **result)
    return heuristic_prediction(customer, strategy)

print(f"🔑 Modo: {'API REAL (Claude Haiku)' if USE_REAL_LLM else 'MOCK — defina ANTHROPIC_API_KEY no ambiente pra usar a API real'}")

🔑 Modo: MOCK — defina ANTHROPIC_API_KEY no ambiente pra usar a API real


**Resultado esperado:** `🔑 Modo: MOCK — defina ANTHROPIC_API_KEY...` (a
menos que você tenha configurado a chave). Nenhuma chamada de rede acontece
no modo mock.

### 6. Loop agêntico com stopping rule (Semana 5-6)

**Por que esta função existe:** é a "Loop Engineering" da arquitetura do
curso — em vez de aceitar a primeira resposta do LLM, o agente tenta até 3
estratégias diferentes (prompts diferentes, reais ou mockados) até a
confiança bater 0.85, ou até esgotar as tentativas. Sem essa regra de
parada, ou o agente para cedo demais (resposta ruim) ou continua pra
sempre (custo infinito).

In [7]:
def churn_agentic_loop(customer: CustomerState, min_confidence: float = 0.85, max_iterations: int = 3) -> CustomerState:
    """Loop Engineering: decide -> act -> observe -> repete até confiança
    suficiente ou esgotar o orçamento de iterações (stopping rule)."""
    strategies = ["heuristic", "cot", "rag_augmented"]
    for i in range(max_iterations):
        customer.strategy_attempts = i + 1
        strategy = strategies[min(i, len(strategies) - 1)]
        prediction = mock_llm_call(customer, strategy)
        customer.prediction = prediction
        print(f"  [{customer.customer_id}] tentativa {i+1} ({strategy}): confidence={prediction.confidence}")
        if prediction.confidence >= min_confidence:
            break
    return customer

**Resultado esperado:** ao rodar o grafo (próxima seção), você vai ver 1-3
linhas por cliente do tipo `[cust_0000] tentativa 1 (heuristic):
confidence=0.7X` — o número de tentativas varia por cliente, dependendo de
quando a confiança cruza 0.85 (no mock, raramente chega lá com 1 tentativa
só, então a maioria roda as 3 estratégias).

### 7. Grafo (Semana 6-7): Gather → Retrieve → Predict → Verify → Recommend

**Por que esta estrutura existe:** em vez de uma função só fazendo tudo, o
LangGraph estrutura o pipeline em 4 nós explícitos. Cada nó tem uma
responsabilidade única — isso torna o fluxo visualizável e testável nó a
nó (Semana 6), e é onde a Adaptive RAG (`node_retrieve`) e o Verifier do
Harness (`node_verify`) realmente se encaixam na execução.

In [8]:
from langgraph.graph import StateGraph, START, END

def node_gather(state: CustomerState) -> CustomerState:
    return state

def node_retrieve(state: CustomerState) -> CustomerState:
    state.similar_churned = retrieve_similar_churned(state, CHURN_HISTORY)
    return state

def node_predict(state: CustomerState) -> CustomerState:
    return churn_agentic_loop(state)

def node_verify(state: CustomerState) -> CustomerState:
    """Harness Verifier: escalona pra humano se confiança ficar baixa mesmo
    após esgotar as estratégias."""
    if state.prediction and state.prediction.confidence < 0.60:
        state.prediction.recommended_action = "ESCALAR PARA HUMANO: confiança baixa"
    state.verified = True
    return state

graph = StateGraph(CustomerState)
graph.add_node("gather", node_gather)
graph.add_node("retrieve", node_retrieve)
graph.add_node("predict", node_predict)
graph.add_node("verify", node_verify)
graph.add_edge(START, "gather")
graph.add_edge("gather", "retrieve")
graph.add_edge("retrieve", "predict")
graph.add_edge("predict", "verify")
graph.add_edge("verify", END)

churn_agent = graph.compile()

**Resultado esperado:** nenhum output — só compila o grafo. `churn_agent`
agora é um objeto executável (`.invoke()`).

### 8. Rodando ponta a ponta

Nota: `churn_agent.invoke(c)` retorna um **novo** state — não muta `c`
original. Guardamos os resultados numa lista pra usar depois (armadilha
comum com LangGraph, documentada na Semana 6).

In [9]:
results = []
for c in customers[:5]:
    result = churn_agent.invoke(c)
    p = result["prediction"] if isinstance(result, dict) else result.prediction
    print(f"→ {c.customer_id}: churn={p.will_churn} conf={p.confidence} ação='{p.recommended_action}'\n")
    results.append(result)

  [cust_0000] tentativa 1 (heuristic): confidence=0.59
  [cust_0000] tentativa 2 (cot): confidence=0.67
  [cust_0000] tentativa 3 (rag_augmented): confidence=0.74
→ cust_0000: churn=False conf=0.74 ação='monitorar'

  [cust_0001] tentativa 1 (heuristic): confidence=0.73
  [cust_0001] tentativa 2 (cot): confidence=0.81
  [cust_0001] tentativa 3 (rag_augmented): confidence=0.88
→ cust_0001: churn=True conf=0.88 ação='oferecer desconto + onboarding assistido'

  [cust_0002] tentativa 1 (heuristic): confidence=0.53
  [cust_0002] tentativa 2 (cot): confidence=0.61
  [cust_0002] tentativa 3 (rag_augmented): confidence=0.68
→ cust_0002: churn=False conf=0.68 ação='monitorar'

  [cust_0003] tentativa 1 (heuristic): confidence=0.7
  [cust_0003] tentativa 2 (cot): confidence=0.78
  [cust_0003] tentativa 3 (rag_augmented): confidence=0.85
→ cust_0003: churn=True conf=0.85 ação='oferecer desconto + onboarding assistido'

  [cust_0004] tentativa 1 (heuristic): confidence=0.65
  [cust_0004] tentativ

**Resultado esperado:** 5 blocos, um por cliente — cada um mostra 1-3 linhas
de tentativa (da seção 6) seguidas da linha `→ cust_XXXX: churn=True/False
conf=0.XX ação='...'`. Como os dados são sintéticos e a seed é fixa, os
mesmos 5 clientes e resultados aparecem toda vez que você roda no modo mock.

### 9. Observability mínima (Semana 8)

**Por que esta função existe:** em produção, você não lê print — você lê
logs estruturados num sistema tipo Cloud Logging. `log_prediction` simula
esse formato (dict com campos fixos), pronto pra virar um `logger.info(...)`
de verdade.

In [10]:
def log_prediction(result):
    """Structured log — em produção isso vai pro Cloud Logging (GCP)."""
    get = (lambda k: result[k]) if isinstance(result, dict) else (lambda k: getattr(result, k))
    p = get("prediction")
    print({
        "event": "churn_prediction",
        "customer_id": get("customer_id"),
        "will_churn": p.will_churn,
        "confidence": p.confidence,
        "iterations": get("strategy_attempts"),
        "escalated": "ESCALAR" in p.recommended_action,
    })

for r in results[:3]:
    log_prediction(r)

{'event': 'churn_prediction', 'customer_id': 'cust_0000', 'will_churn': False, 'confidence': 0.74, 'iterations': 3, 'escalated': False}
{'event': 'churn_prediction', 'customer_id': 'cust_0001', 'will_churn': True, 'confidence': 0.88, 'iterations': 3, 'escalated': False}
{'event': 'churn_prediction', 'customer_id': 'cust_0002', 'will_churn': False, 'confidence': 0.68, 'iterations': 3, 'escalated': False}


**Resultado esperado:** 3 dicts impressos, um por cliente, com os campos
`event`, `customer_id`, `will_churn`, `confidence`, `iterations`,
`escalated` — o formato que um agregador de log (Cloud Logging, Datadog)
consegue indexar e filtrar.

### 10. Testes básicos (Semana 9)

**Por que esta célula existe:** valida invariantes do sistema — coisas que
*sempre* devem ser verdade, independente do dado de entrada. Não é uma
suite completa (isso é trabalho de produção real), mas é mais do que "rodou
sem erro": confirma que o schema é respeitado e que a stopping rule
realmente para.

In [11]:
def test_confidence_in_range():
    for r in results:
        p = r["prediction"] if isinstance(r, dict) else r.prediction
        assert 0.0 <= p.confidence <= 1.0, f"confidence fora do range: {p.confidence}"
    print("✓ test_confidence_in_range passou")

def test_stopping_rule_respects_max_iterations():
    for r in results:
        attempts = r["strategy_attempts"] if isinstance(r, dict) else r.strategy_attempts
        assert 1 <= attempts <= 3, f"iterations fora do esperado: {attempts}"
    print("✓ test_stopping_rule_respects_max_iterations passou")

def test_low_confidence_escalates():
    for r in results:
        p = r["prediction"] if isinstance(r, dict) else r.prediction
        if p.confidence < 0.60:
            assert "ESCALAR" in p.recommended_action
    print("✓ test_low_confidence_escalates passou")

test_confidence_in_range()
test_stopping_rule_respects_max_iterations()
test_low_confidence_escalates()

✓ test_confidence_in_range passou
✓ test_stopping_rule_respects_max_iterations passou
✓ test_low_confidence_escalates passou


**Resultado esperado:** 3 linhas `✓ ... passou`. Se qualquer invariante
quebrar, o `assert` levanta `AssertionError` com a mensagem específica —
mais fácil de debugar do que "algo deu errado".

**Próximos passos pra produção:**
- Já dá pra usar Claude de verdade — só definir `ANTHROPIC_API_KEY`
- Trocar `retrieve_similar_churned` por um vector DB real (Chroma/Pinecone)
- Persistir `CustomerState` no Firestore
- Deploy no Cloud Run (ver [`Dockerfile`](./Dockerfile) neste projeto e `docs/source-material/08-plano-estudos-gcp.md`)
- Avaliação real: comparar `will_churn` previsto vs churn observado após 30 dias